# Lip Reading Dataset ETL Pipeline
Checkpoint-safe, retry-enabled pipeline with progress tracking.

In [1]:
import os, sys, json, subprocess, csv
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if (PROJECT_ROOT / "dataset_pipeline" / "pipeline_utils.py").exists():
    PIPELINE_DIR = PROJECT_ROOT / "dataset_pipeline"
else:
    PIPELINE_DIR = PROJECT_ROOT

sys.path.insert(0, str(PIPELINE_DIR))
from pipeline_utils import CheckpointManager, ensure_dir, save_json

PIPELINE_NAME = "lipsynth_pipeline"
CHECKPOINT_DIR = PIPELINE_DIR / "checkpoints"
DATA_DIR = PIPELINE_DIR / "data"

INPUT_CSV = DATA_DIR / "links" / "video_links.csv"
RAW_VIDEOS_DIR = DATA_DIR / "raw_videos"
SEGMENTS_DIR = DATA_DIR / "segments"
LIP_ROIS_DIR = DATA_DIR / "lip_rois"
FINAL_DIR = DATA_DIR / "dataset_final"

PLAYLIST_ID = "PLsRNoUx8w3rPxNGCQYBPobGxNj1BfDT7P"
PLAYLIST_NUM_VIDEOS = 3
WHISPER_MODEL = "base"

for d in [CHECKPOINT_DIR, DATA_DIR, RAW_VIDEOS_DIR, SEGMENTS_DIR, LIP_ROIS_DIR, FINAL_DIR]:
    ensure_dir(d)

checkpoint = CheckpointManager(PIPELINE_NAME, str(CHECKPOINT_DIR))
print(f"Pipeline dir: {PIPELINE_DIR}")
print(f"Data dir: {DATA_DIR}")


Pipeline dir: /home/shravan/Workspace/LipSynth/dataset_pipeline
Data dir: /home/shravan/Workspace/LipSynth/dataset_pipeline/data


In [2]:
import cv2, numpy as np
if checkpoint.load() is None:
    checkpoint.init_state(input_config={"input_csv": str(INPUT_CSV)})
print("Completed steps:", checkpoint.get_completed_steps())


Completed steps: ['step0_fetch_playlist', 'step1_download', 'step2_segment']


## Full Pipeline Run (Recommended First)
Run this section first for an end-to-end dataset build. Use step-by-step cells below only when debugging or re-running a single stage.

In [ ]:
def _run_step_cmd(step_name, cmd):
    print(f"\n=== Running {step_name} ===")
    print("$", " ".join(cmd))
    result = subprocess.run(cmd, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        if result.stderr:
            print(result.stderr)
        checkpoint.save(step_name=step_name, status="failed", step_data={"return_code": result.returncode})
        raise RuntimeError(f"{step_name} failed")


def run_full_pipeline(force_reset=False, start_from=0):
    """Run all pipeline stages in order.

    Args:
        force_reset: If True, clears checkpoint state before running.
        start_from: Step index to start from (0..4).
    """
    if force_reset:
        checkpoint.clear()
        checkpoint.init_state(input_config={"input_csv": str(INPUT_CSV)})

    steps = [
        (
            "step0_fetch_playlist",
            [
                sys.executable,
                str(PIPELINE_DIR / "00_fetch_playlist.py"),
                "--playlist_id", PLAYLIST_ID,
                "--num_videos", str(PLAYLIST_NUM_VIDEOS),
                "--output", str(INPUT_CSV),
            ],
        ),
        (
            "step1_download",
            [
                sys.executable,
                str(PIPELINE_DIR / "01_download_videos.py"),
                "--input", str(INPUT_CSV),
                "--output_dir", str(RAW_VIDEOS_DIR),
            ],
        ),
        (
            "step2_segment",
            [
                sys.executable,
                str(PIPELINE_DIR / "02_segment_clips.py"),
                "--input_dir", str(RAW_VIDEOS_DIR),
                "--output_dir", str(SEGMENTS_DIR),
                "--whisper_model", WHISPER_MODEL,
            ],
        ),
        (
            "step3_visual_features",
            [
                sys.executable,
                str(PIPELINE_DIR / "03_extract_visual_features.py"),
                "--input_dir", str(SEGMENTS_DIR),
                "--output_dir", str(LIP_ROIS_DIR),
                "--detector", "pytorch",
                "--device", "auto",
                "--no_compress",
            ],
        ),
        (
            "step4_finalize",
            [
                sys.executable,
                str(PIPELINE_DIR / "04_finalize_dataset.py"),
                "--segments_dir", str(SEGMENTS_DIR),
                "--lip_rois_dir", str(LIP_ROIS_DIR),
                "--output_dir", str(FINAL_DIR),
            ],
        ),
    ]

    for idx, (step_name, cmd) in enumerate(steps):
        if idx < start_from:
            continue
        checkpoint.save(step_name=step_name, status="running")
        _run_step_cmd(step_name, cmd)
        checkpoint.save(step_name=step_name, status="completed")

    print("\nPipeline complete.")
    print("Completed steps:", checkpoint.get_completed_steps())


# Example usage:
# run_full_pipeline()                     # normal full run
# run_full_pipeline(force_reset=True)     # full run from clean checkpoint
# run_full_pipeline(start_from=2)         # resume from Step 2


## Reset Helpers (Run Before Manual Re-runs)
Use these helpers to reset one stage or the entire pipeline checkpoint before executing step-by-step cells.

In [ ]:
def reset_step(step_num):
    step_names = {
        0: "step0_fetch_playlist",
        1: "step1_download",
        2: "step2_segment",
        3: "step3_visual_features",
        4: "step4_finalize",
    }
    if step_num not in step_names:
        raise ValueError("step_num must be one of: 0, 1, 2, 3, 4")
    checkpoint.reset_step(step_names[step_num])
    print(f"Step {step_num} reset ({step_names[step_num]})")


def reset_all(confirm=False):
    if not confirm:
        print("Pass confirm=True to clear the full pipeline checkpoint.")
        return
    checkpoint.clear()
    checkpoint.init_state(input_config={"input_csv": str(INPUT_CSV)})
    print("Pipeline checkpoint reset.")


# Example usage:
# reset_step(3)
# reset_all(confirm=True)


## Step-by-Step Runs (Optional)
Use the following cells when you need to debug or rerun specific stages after using reset helpers.

## Step 0: Fetch Playlist

In [3]:
def csv_row_count(csv_path: Path) -> int:
    if not csv_path.exists():
        return 0
    with open(csv_path, "r") as f:
        return len(list(csv.DictReader(f)))


def run_step0_script():
    cmd = [
        sys.executable,
        str(PIPELINE_DIR / "00_fetch_playlist.py"),
        "--playlist_id", PLAYLIST_ID,
        "--num_videos", str(PLAYLIST_NUM_VIDEOS),
        "--output", str(INPUT_CSV),
    ]
    result = subprocess.run(cmd, text=True, capture_output=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError("Step 0 failed: could not refresh data/links/video_links.csv")


STEP_NAME = "step0_fetch_playlist"
rows_before = csv_row_count(INPUT_CSV)

if checkpoint.is_step_completed(STEP_NAME) and rows_before >= PLAYLIST_NUM_VIDEOS:
    print(f"Step 0 already completed ({rows_before} rows in CSV)")
else:
    if checkpoint.is_step_completed(STEP_NAME) and rows_before < PLAYLIST_NUM_VIDEOS:
        print(f"Step 0 marked complete but CSV has only {rows_before} rows. Re-running...")
        checkpoint.save(step_name=STEP_NAME, status="pending", step_data={"reason": "csv_row_count_too_low"})

    checkpoint.save(step_name=STEP_NAME, status="running", step_data={"target_rows": PLAYLIST_NUM_VIDEOS})
    run_step0_script()

    rows_after = csv_row_count(INPUT_CSV)
    if rows_after < PLAYLIST_NUM_VIDEOS:
        checkpoint.save(step_name=STEP_NAME, status="failed", step_data={"row_count": rows_after})
        raise RuntimeError(f"Step 0 failed. Expected >= {PLAYLIST_NUM_VIDEOS} rows, found {rows_after}")

    checkpoint.save(step_name=STEP_NAME, status="completed", step_data={"row_count": rows_after, "csv": str(INPUT_CSV)})
    print(f"Step 0 completed ({rows_after} rows)")


Step 0 already completed (3 rows in CSV)


## Step 1: Download Videos

In [4]:
def has_complete_download(speaker_id: str) -> bool:
    speaker_dir = RAW_VIDEOS_DIR / speaker_id
    video_path = speaker_dir / "full_video.mp4"
    audio_path = speaker_dir / "full_audio.wav"
    return (
        video_path.exists() and audio_path.exists() and
        video_path.stat().st_size > 1024 and audio_path.stat().st_size > 1024
    )


def expected_speaker_ids():
    with open(INPUT_CSV, "r") as f:
        return [row["speaker_id"] for row in csv.DictReader(f)]


def run_step1_script():
    cmd = [
        sys.executable,
        str(PIPELINE_DIR / "01_download_videos.py"),
        "--input", str(INPUT_CSV),
        "--output_dir", str(RAW_VIDEOS_DIR),
    ]
    result = subprocess.run(cmd, text=True, capture_output=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError("Step 1 failed: download artifacts incomplete")


STEP_NAME = "step1_download"
expected = expected_speaker_ids()
missing_before = [sid for sid in expected if not has_complete_download(sid)]

if checkpoint.is_step_completed(STEP_NAME) and not missing_before:
    print("Step 1 already completed")
else:
    if checkpoint.is_step_completed(STEP_NAME) and missing_before:
        print(f"Step 1 marked complete but missing outputs for: {missing_before}. Re-running...")
        checkpoint.save(step_name=STEP_NAME, status="pending", step_data={"reason": "missing_outputs_detected"})

    checkpoint.save(step_name=STEP_NAME, status="running", step_data={"expected_speakers": expected})
    run_step1_script()

    missing_after = [sid for sid in expected if not has_complete_download(sid)]
    if missing_after:
        checkpoint.save(step_name=STEP_NAME, status="failed", step_data={"missing_speakers": missing_after})
        raise RuntimeError(f"Step 1 failed. Missing complete downloads for: {missing_after}")

    checkpoint.save(step_name=STEP_NAME, status="completed", step_data={"completed_speakers": expected})
    print("Step 1 completed")


Step 1 already completed


## Step 2: Segment Clips

In [5]:
def clip_artifacts_complete(seg_dir: Path, clip_id: str) -> bool:
    checks = [
        (seg_dir / f"{clip_id}.mp4", 1024),
        (seg_dir / f"{clip_id}.wav", 1024),
        (seg_dir / f"{clip_id}.txt", 1),
        (seg_dir / f"{clip_id}.json", 16),
    ]
    return all(p.exists() and p.stat().st_size >= min_size for p, min_size in checks)


def count_complete_segment_clips(speaker_id: str) -> int:
    seg_dir = SEGMENTS_DIR / speaker_id
    if not seg_dir.exists():
        return 0
    count = 0
    for mp4 in seg_dir.glob("*.mp4"):
        clip_id = mp4.stem
        if clip_artifacts_complete(seg_dir, clip_id):
            count += 1
    return count


def speakers_ready_for_segmentation():
    ready = []
    for d in sorted(RAW_VIDEOS_DIR.iterdir()):
        if d.is_dir() and (d / "full_video.mp4").exists() and (d / "full_audio.wav").exists():
            ready.append(d.name)
    return ready


def run_step2_script():
    cmd = [
        sys.executable,
        str(PIPELINE_DIR / "02_segment_clips.py"),
        "--input_dir", str(RAW_VIDEOS_DIR),
        "--output_dir", str(SEGMENTS_DIR),
        "--whisper_model", WHISPER_MODEL,
    ]
    result = subprocess.run(cmd, text=True, capture_output=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError("Step 2 failed: segmentation incomplete")


STEP_NAME = "step2_segment"
expected = speakers_ready_for_segmentation()
if not expected:
    raise RuntimeError("No speakers with complete raw artifacts. Run Step 1 first.")

missing_before = [sid for sid in expected if count_complete_segment_clips(sid) == 0]

if checkpoint.is_step_completed(STEP_NAME) and not missing_before:
    print("Step 2 already completed")
else:
    if checkpoint.is_step_completed(STEP_NAME) and missing_before:
        print(f"Step 2 marked complete but missing segments for: {missing_before}. Re-running...")
        checkpoint.save(step_name=STEP_NAME, status="pending", step_data={"reason": "missing_outputs_detected"})

    checkpoint.save(step_name=STEP_NAME, status="running", step_data={"expected_speakers": expected})
    run_step2_script()

    missing_after = [sid for sid in expected if count_complete_segment_clips(sid) == 0]
    if missing_after:
        checkpoint.save(step_name=STEP_NAME, status="failed", step_data={"missing_speakers": missing_after})
        raise RuntimeError(f"Step 2 failed. Missing segment clips for: {missing_after}")

    summary = {sid: count_complete_segment_clips(sid) for sid in expected}
    checkpoint.save(step_name=STEP_NAME, status="completed", step_data={"clip_counts": summary})
    print("Step 2 completed")


Step 2 already completed


## Step 3: Extract Visual ROIs + Features

In [8]:
def has_roi(speaker_id: str, clip_id: str) -> bool:
    roi_path = LIP_ROIS_DIR / speaker_id / f"{clip_id}.npz"
    return roi_path.exists() and roi_path.stat().st_size > 256


def missing_roi_clips(show_progress: bool = False):
    missing = []
    clips = []
    rejected = {}
    if LIP_ROIS_DIR.exists():
        for roi_speaker_dir in sorted(LIP_ROIS_DIR.iterdir()):
            if not roi_speaker_dir.is_dir():
                continue
            report_path = roi_speaker_dir / "extraction_report.json"
            if not report_path.exists():
                continue
            try:
                report = json.loads(report_path.read_text())
                rejected[roi_speaker_dir.name] = {
                    clip.get("clip_id")
                    for clip in report.get("clips", [])
                    if clip.get("status") == "rejected" and clip.get("clip_id")
                }
            except Exception:
                continue
    for speaker_dir in sorted(SEGMENTS_DIR.iterdir()):
        if not speaker_dir.is_dir():
            continue
        speaker_id = speaker_dir.name
        for mp4 in speaker_dir.glob("*.mp4"):
            clips.append((speaker_id, mp4))

    clip_iter = clips
    if show_progress:
        try:
            from tqdm.auto import tqdm
            clip_iter = tqdm(clips, desc="Step 3 ROI check", unit="clip")
        except ImportError:
            pass

    for speaker_id, mp4 in clip_iter:
        clip_id = mp4.stem
        if clip_id in rejected.get(speaker_id, set()):
            continue
        if not has_roi(speaker_id, clip_id):
            missing.append((speaker_id, clip_id))
    return missing


SAVE_STEP3_FRAMES = False  # set True only for debugging frame-level crops


def run_step3_script():
    cmd = [
        sys.executable,
        str(PIPELINE_DIR / "03_extract_visual_features.py"),
        "--input_dir", str(SEGMENTS_DIR),
        "--output_dir", str(LIP_ROIS_DIR),
        "--detector", "pytorch",
        "--device", "auto",
        "--no_compress",
    ]
    if SAVE_STEP3_FRAMES:
        cmd.append("--save_frames")

    process = subprocess.Popen(
        cmd,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    output_lines = []
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
        output_lines.append(line)
    return_code = process.wait()
    if return_code != 0:
        tail = "".join(output_lines[-40:]).strip()
        details = f"\nLast logs:\n{tail}" if tail else ""
        raise RuntimeError(f"Step 3 failed (exit code {return_code}).{details}")


STEP_NAME = "step3_visual_features"
if not SEGMENTS_DIR.exists() or not any(SEGMENTS_DIR.rglob("*.mp4")):
    raise RuntimeError("No segmented clips found. Run Step 2 first.")

missing_before = missing_roi_clips(show_progress=True)
if checkpoint.is_step_completed(STEP_NAME) and not missing_before:
    print("Step 3 already completed")
else:
    if checkpoint.is_step_completed(STEP_NAME) and missing_before:
        print(f"Step 3 marked complete but {len(missing_before)} ROI files are missing. Re-running...")
        checkpoint.save(step_name=STEP_NAME, status="pending", step_data={"reason": "missing_outputs_detected"})

    checkpoint.save(step_name=STEP_NAME, status="running")
    run_step3_script()

    missing_after = missing_roi_clips(show_progress=True)
    if missing_after:
        checkpoint.save(step_name=STEP_NAME, status="failed", step_data={"missing_count": len(missing_after), "examples": missing_after[:10]})
        raise RuntimeError(f"Step 3 failed. Missing ROI files: {len(missing_after)}")

    checkpoint.save(step_name=STEP_NAME, status="completed", step_data={"roi_count": len(list(LIP_ROIS_DIR.rglob('*.npz')))})
    print("Step 3 completed")

Step 3 ROI check:   0%|          | 0/338 [00:00<?, ?clip/s]

Using pytorch face detector on device=cuda
Found 3 speakers to process


Step 3 speakers:   0%|          | 0/3 [00:00<?, ?speaker/s][1/3] Processing spk_001...


spk_001:   0%|          | 0/109 [00:00<?, ?clip/s]

spk_001:   1%|          | 1/109 [00:29<52:24, 29.12s/clip]

spk_001:   2%|▏         | 2/109 [00:49<42:44, 23.97s/clip]

spk_001:   3%|▎         | 3/109 [01:13<42:26, 24.02s/clip]

spk_001:   4%|▎         | 4/109 [01:27<35:23, 20.22s/clip]

spk_001:   5%|▍         | 5/109 [01:40<30:06, 17.37s/clip]

spk_001:   6%|▌         | 6/109 [01:58<30:19, 17.66s/clip]

spk_001:   6%|▋         | 7/109 [02:09<26:27, 15.57s/clip]/home/shravan/Workspace/LipSynth/.venv/lib/python3.12/site-packages/face_alignment/api.py:147: UserWarning: No faces were detected.
  warnings.warn("No faces were detected.")


spk_001:   9%|▉         | 10/109 [02:14<12:10,  7.38s/clip]

spk_001:  11%|█         | 12/109 [02:26<11:11,  6.92s/clip]

spk_001:  12%|█▏        | 13/109 [02:41<13:47,  8.62s/clip]

spk_001:

Step 3 ROI check:   0%|          | 0/338 [00:00<?, ?clip/s]

Step 3 completed


## Step 4: Finalize Dataset

In [9]:
def final_outputs_ok() -> bool:
    required = [FINAL_DIR / "train.tsv", FINAL_DIR / "val.tsv", FINAL_DIR / "test.tsv", FINAL_DIR / "dataset_stats.json"]
    if not all(p.exists() and p.stat().st_size > 0 for p in required):
        return False
    try:
        stats = json.loads((FINAL_DIR / "dataset_stats.json").read_text())
        return stats.get("overall", {}).get("total_clips", 0) > 0
    except Exception:
        return False


def run_step4_script():
    cmd = [
        sys.executable,
        str(PIPELINE_DIR / "04_finalize_dataset.py"),
        "--segments_dir", str(SEGMENTS_DIR),
        "--lip_rois_dir", str(LIP_ROIS_DIR),
        "--output_dir", str(FINAL_DIR),
    ]
    result = subprocess.run(cmd, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        if result.stderr:
            print(result.stderr)
        details = (result.stderr or result.stdout or "").strip()
        tail = "\n".join(details.splitlines()[-40:]) if details else ""
        extra = f"\nLast logs:\n{tail}" if tail else ""
        raise RuntimeError(f"Step 4 failed: final dataset incomplete.{extra}")


STEP_NAME = "step4_finalize"
if checkpoint.is_step_completed(STEP_NAME) and final_outputs_ok():
    print("Step 4 already completed")
else:
    if checkpoint.is_step_completed(STEP_NAME) and not final_outputs_ok():
        print("Step 4 marked complete but final outputs are missing/broken. Re-running...")
        checkpoint.save(step_name=STEP_NAME, status="pending", step_data={"reason": "missing_outputs_detected"})

    checkpoint.save(step_name=STEP_NAME, status="running")
    run_step4_script()

    if not final_outputs_ok():
        checkpoint.save(step_name=STEP_NAME, status="failed", step_data={"reason": "final_outputs_incomplete"})
        raise RuntimeError("Step 4 failed: final outputs not valid")

    stats = json.loads((FINAL_DIR / "dataset_stats.json").read_text())
    checkpoint.save(step_name=STEP_NAME, status="completed", step_data={"stats": stats})
    print("Step 4 completed")
    print(f"Dataset ready at: {FINAL_DIR}")


Checking for missing artifacts...
  Found 54 incomplete clips before finalization.
  These clips will be excluded from the final dataset.
  Example: [{'clip_id': 'spk_001_0001', 'speaker_id': 'spk_001', 'missing': ['roi']}, {'clip_id': 'spk_001_0002', 'speaker_id': 'spk_001', 'missing': ['roi']}, {'clip_id': 'spk_001_0003', 'speaker_id': 'spk_001', 'missing': ['roi']}, {'clip_id': 'spk_001_0004', 'speaker_id': 'spk_001', 'missing': ['roi']}, {'clip_id': 'spk_001_0005', 'speaker_id': 'spk_001', 'missing': ['roi']}]
Gathering valid clips...
  Found 284 valid clips

Creating speaker-disjoint splits...
  Split by speaker:
    Train: 1 speakers, 101 clips
    Val:   1 speakers, 75 clips
    Test:  1 speakers, 108 clips

Writing manifest files...
  /home/shravan/Workspace/LipSynth/dataset_pipeline/data/dataset_final/train.tsv
  /home/shravan/Workspace/LipSynth/dataset_pipeline/data/dataset_final/val.tsv
  /home/shravan/Workspace/LipSynth/dataset_pipeline/data/dataset_final/test.tsv

Organizi